In [5]:
from transformers import AutoTokenizer

In [1]:
%%bash
pip install git+https://github.com/HinoEji/LangChain-Transformer-ChatModel.git

  Cloning https://github.com/HinoEji/LangChain-Transformer-ChatModel.git to /tmp/pip-req-build-lfdhscht
  Resolved https://github.com/HinoEji/LangChain-Transformer-ChatModel.git to commit 70b71b0baac2e7c5cabcb4a595742b177cd9ffc1
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'


  Running command git clone --filter=blob:none --quiet https://github.com/HinoEji/LangChain-Transformer-ChatModel.git /tmp/pip-req-build-lfdhscht


In [1]:
from transformer_chat_model import TransformerChatModel
from transformers import BitsAndBytesConfig
import torch
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
import datetime
import random
from typing import Dict


**Chat Model**
- With popular GPU such as P100 or T4 (x2 in Kaggle) + quantization we can easily load LLMs with 8B parameters
- I strongly recommend using models with at least 7B parameters to ensure sufficient reasoning capacity for adhering to internal rules, thereby improving the stability of the tool-calling process.


In [ ]:
pretrained_model_name_or_path = "Qwen/Qwen2.5-7B-Instruct-1M"
# Create your quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)
chat_model = TransformerChatModel(
    pretrained_model_name_or_path=pretrained_model_name_or_path,
    quantization_config = bnb_config,
    max_new_tokens = 1024,
    device = "auto"
    )

In [ ]:
bnb_config[""]

BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "bfloat16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": false,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}

In [21]:
# normal inference
response = chat_model.invoke("what is the probability of getting a 6 when rolling a dice?")
response

AIMessage(content="The probability of rolling a 6 on a fair six-sided die is \\( \\frac{1}{6} \\).\n\nHere's why: A standard die has six faces, each with an equal chance of landing face up. Since there is only one face with a 6 on it, the probability is:\n\n\\[ P(\\text{rolling a 6}) = \\frac{\\text{Number of favorable outcomes}}{\\text{Total number of possible outcomes}} = \\frac{1}{6} \\]\n\nSo, the probability is approximately 0.1667 or 16.67%.", additional_kwargs={}, response_metadata={}, id='lc_run--019c9f16-4e82-7d13-af9b-ce753b0f768c-0', tool_calls=[], invalid_tool_calls=[])

- Tool Calling with chat model

In [23]:
@tool
def roll_dice(n: int) -> Dict[int, int]:
    """
    Roll a 6-faced dice n times.
    Returns a dictionary where:
    - key: dice face (1-6)
    - value: number of occurrences
    """
    if n <= 0:
        raise ValueError("n must be a positive integer")

    # Initialize all faces with 0
    results = {i: 0 for i in range(1, 7)}

    for _ in range(n):
        face = random.randint(1, 6)
        results[face] += 1

    return results


@tool
def calculate_probability(x: int, N: int) -> float:
    """
    Calculate probability as x / N.
    x: number of successful outcomes
    N: total number of trials
    """
    if N <= 0:
        raise ValueError("Total number of trials N must be positive")

    if x < 0:
        raise ValueError("x must be non-negative")

    return x / N

In [24]:
# bind tool 
chat_model_with_tool = chat_model.bind_tools(tools=[roll_dice, calculate_probability])

In [26]:
response = chat_model_with_tool.invoke("what is the probability of getting a 6 when rolling dice?")
response

AIMessage(content='', additional_kwargs={}, response_metadata={}, id='lc_run--019c9f18-a5ca-7f33-b862-ca9f7c971456-0', tool_calls=[{'name': 'roll_dice', 'args': {'n': 1000}, 'id': 'lc_c385c3d4-72d7-4c5b-9780-ccb720e20882', 'type': 'tool_call'}], invalid_tool_calls=[])

**LangChain Agent**

In [27]:
agent = create_agent(
    model = chat_model,
    tools = [roll_dice, calculate_probability],
    system_prompt="You are a helpful assistant."
)

In [32]:
messages = [
    HumanMessage(content = "Condunct a experiment to find the probability of getting a 6 when rolling dice with high reliability?")
]
response = agent.invoke({"messages":messages})
response

{'messages': [HumanMessage(content='Condunct a experiment to find the probability of getting a 6 when rolling dice with high reliability?', additional_kwargs={}, response_metadata={}, id='51a4dc85-9870-42b9-b965-9a8e52ced48c'),
  AIMessage(content='', additional_kwargs={}, response_metadata={}, id='lc_run--019c9f24-d07a-74e1-8d1f-482ec4c4fe77-0', tool_calls=[{'name': 'roll_dice', 'args': {'n': 1000}, 'id': 'lc_c10e97e2-0156-4227-8059-d242c64af5cc', 'type': 'tool_call'}], invalid_tool_calls=[]),
  ToolMessage(content='{"1": 183, "2": 167, "3": 168, "4": 179, "5": 147, "6": 156}', name='roll_dice', id='970faa59-d724-4bad-90cc-4bde3d874575', tool_call_id='lc_c10e97e2-0156-4227-8059-d242c64af5cc'),
  AIMessage(content='', additional_kwargs={}, response_metadata={}, id='lc_run--019c9f24-d51e-7da2-9468-b8a3000cfef3-0', tool_calls=[{'name': 'calculate_probability', 'args': {'x': 156, 'N': 1000}, 'id': 'lc_d9d037df-4666-4bb1-9f5d-0d2d66774823', 'type': 'tool_call'}], invalid_tool_calls=[]),
  

In [33]:
for i in response["messages"]:
    i.pretty_print()

================================ Human Message =================================

Condunct a experiment to find the probability of getting a 6 when rolling dice with high reliability?
================================== Ai Message ==================================
Tool Calls:
  roll_dice (lc_c10e97e2-0156-4227-8059-d242c64af5cc)
 Call ID: lc_c10e97e2-0156-4227-8059-d242c64af5cc
  Args:
    n: 1000
================================= Tool Message =================================
Name: roll_dice

{"1": 183, "2": 167, "3": 168, "4": 179, "5": 147, "6": 156}
================================== Ai Message ==================================
Tool Calls:
  calculate_probability (lc_d9d037df-4666-4bb1-9f5d-0d2d66774823)
 Call ID: lc_d9d037df-4666-4bb1-9f5d-0d2d66774823
  Args:
    x: 156
    N: 1000
================================= Tool Message =================================
Name: calculate_probability

0.156
================================== Ai Message ==================================

Af

**Stream**

In [ ]:
# update mode
for step in agent.stream(
    {"messages": messages},
      stream_mode = "updates"
):
    print(step)

{'model': {'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={}, id='lc_run--019c9f27-360a-7a50-ad1c-ed400f196eb6-0', tool_calls=[{'name': 'roll_dice', 'args': {'n': 10000}, 'id': 'lc_5260b83b-c6d7-4889-b2ae-768c1f5d7746', 'type': 'tool_call'}], invalid_tool_calls=[])]}}
{'tools': {'messages': [ToolMessage(content='{"1": 1640, "2": 1608, "3": 1675, "4": 1737, "5": 1724, "6": 1616}', name='roll_dice', id='746a1137-a693-4e6a-9136-5d7877de3a6f', tool_call_id='lc_5260b83b-c6d7-4889-b2ae-768c1f5d7746')]}}
{'model': {'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={}, id='lc_run--019c9f27-38c7-7392-ae77-11320f479cdc-0', tool_calls=[{'name': 'calculate_probability', 'args': {'x': 1616, 'N': 10000}, 'id': 'lc_bd9e639b-3f0b-4a5c-aa77-c24823e5b648', 'type': 'tool_call'}], invalid_tool_calls=[])]}}
{'tools': {'messages': [ToolMessage(content='0.1616', name='calculate_probability', id='0d928d57-cb6b-4fec-afad-cb852a90daf0', tool_call_id='lc_bd9e6

In [43]:
from langchain_core.messages import AIMessageChunk, HumanMessageChunk, ToolMessage

In [46]:
prev_type = None
for step in agent.stream(
    {"messages": messages},
      stream_mode = "messages"
):
    print(step)
        
 
        
    


(AIMessageChunk(content='', additional_kwargs={}, response_metadata={}, id='lc_run--019c9f31-c14d-7082-b7ad-bca2f3759fec', tool_calls=[{'name': '', 'args': {}, 'id': 'lc_3656ff85-c7bb-4931-8be5-5284a4fc6327', 'type': 'tool_call'}], invalid_tool_calls=[], tool_call_chunks=[{'name': None, 'args': None, 'id': 'lc_3656ff85-c7bb-4931-8be5-5284a4fc6327', 'index': 0, 'type': 'tool_call_chunk'}]), {'langgraph_step': 1, 'langgraph_node': 'model', 'langgraph_triggers': ('branch:to:model',), 'langgraph_path': ('__pregel_pull', 'model'), 'langgraph_checkpoint_ns': 'model:4708022c-2075-9193-ec91-8c8ea57720f7', 'checkpoint_ns': 'model:4708022c-2075-9193-ec91-8c8ea57720f7', 'ls_provider': 'transformerchatmodel', 'ls_model_type': 'chat', 'ls_temperature': 0.7})
(AIMessageChunk(content='', additional_kwargs={}, response_metadata={}, id='lc_run--019c9f31-c14d-7082-b7ad-bca2f3759fec', tool_calls=[{'name': 'roll_dice', 'args': {}, 'id': None, 'type': 'tool_call'}], invalid_tool_calls=[], tool_call_chunks=